# 02 — Exploratory Data Analysis & Visualization
**DRM Key Demand Forecasting**

This notebook explores the cleaned daily DRM key data to uncover patterns, seasonality, and trends that inform the forecasting approach.

---

**Analysis areas:**
1. Overall trend and distribution
2. Weekly seasonality pattern
3. Monthly seasonality pattern
4. Source breakdown (IPTV vs Fim+ vs BHD)
5. Time series decomposition
6. Stationarity testing
7. Autocorrelation analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
import os

warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'lines.linewidth': 1.5,
    'axes.grid': True,
    'grid.alpha': 0.3
})
sns.set_palette('husl')

print('Libraries loaded.')

In [ ]:
DATA_DIR = os.path.join('..', 'Dataset')

df = pd.read_csv(
    os.path.join(DATA_DIR, 'daily_drm_keys_clean.csv'),
    parse_dates=['LogDate']
)

df_source = pd.read_csv(
    os.path.join(DATA_DIR, 'daily_drm_keys_by_source.csv'),
    parse_dates=['LogDate']
)

print(f'Loaded: {len(df)} days | {df["LogDate"].min().date()} → {df["LogDate"].max().date()}')
display(df.head())

## 1. Overall Trend & Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Time series plot
ax1 = axes[0]
ax1.plot(df['LogDate'], df['Total_Daily_Keys'], alpha=0.5, label='Daily', color='steelblue')
ax1.plot(df['LogDate'], df['MA_7'], label='7-day MA', color='darkorange', linewidth=2)
ax1.plot(df['LogDate'], df['MA_30'], label='30-day MA', color='crimson', linewidth=2)
ax1.set_title('Daily DRM Key Demand — Trend Overview')
ax1.set_xlabel('Date')
ax1.set_ylabel('DRM Keys Issued')
ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.tick_params(axis='x', rotation=45)

# Distribution
ax2 = axes[1]
ax2.hist(df['Total_Daily_Keys'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(df['Total_Daily_Keys'].mean(), color='crimson', linestyle='--', label=f'Mean: {df["Total_Daily_Keys"].mean():,.0f}')
ax2.axvline(df['Total_Daily_Keys'].median(), color='darkorange', linestyle='--', label=f'Median: {df["Total_Daily_Keys"].median():,.0f}')
ax2.set_title('Distribution of Daily DRM Keys')
ax2.set_xlabel('DRM Keys per Day')
ax2.set_ylabel('Frequency')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
print('Descriptive Statistics — Daily DRM Keys')
print('=' * 45)
stats = df['Total_Daily_Keys'].describe()
stats['IQR'] = stats['75%'] - stats['25%']
stats['CoV'] = df['Total_Daily_Keys'].std() / df['Total_Daily_Keys'].mean() * 100
print(stats.to_string())
print(f'\nCoefficient of Variation: {stats["CoV"]:.1f}%')

## 2. Weekly Seasonality Pattern

Weekday vs weekend consumption — a core driver for DRM key demand.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

weekday_stats = (
    df.groupby('DayOfWeekName')['Total_Daily_Keys']
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .reindex(day_order)
    .round(0)
)

display(weekday_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Boxplot by day of week
ax1 = axes[0]
df_plot = df.copy()
df_plot['DayOfWeekName'] = pd.Categorical(df_plot['DayOfWeekName'], categories=day_order, ordered=True)
sns.boxplot(data=df_plot, x='DayOfWeekName', y='Total_Daily_Keys', ax=ax1, palette='coolwarm')
ax1.set_title('DRM Key Demand by Day of Week')
ax1.set_xlabel('')
ax1.set_ylabel('DRM Keys')
ax1.tick_params(axis='x', rotation=45)

# Mean bar chart with weekend highlight
ax2 = axes[1]
colors = ['#4a90d9' if d in ['Saturday', 'Sunday'] else '#95a5a6' for d in day_order]
ax2.bar(day_order, weekday_stats['mean'], color=colors, edgecolor='white')
ax2.axhline(df['Total_Daily_Keys'].mean(), color='crimson', linestyle='--', alpha=0.7, label='Overall Mean')
ax2.set_title('Average DRM Keys by Weekday (Weekend highlighted)')
ax2.set_ylabel('Avg DRM Keys')
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

plt.tight_layout()
plt.show()

# Weekend vs weekday comparison
weekend_avg = df[df['IsWeekend'] == 1]['Total_Daily_Keys'].mean()
weekday_avg = df[df['IsWeekend'] == 0]['Total_Daily_Keys'].mean()
print(f'Weekend average: {weekend_avg:,.0f} | Weekday average: {weekday_avg:,.0f}')
print(f'Weekend uplift : {((weekend_avg / weekday_avg) - 1) * 100:+.1f}%')

## 3. Monthly Seasonality Pattern

In [ ]:
monthly_stats = (
    df.groupby(['Year', 'Month'])['Total_Daily_Keys']
    .agg(['sum', 'mean', 'max'])
    .reset_index()
)
monthly_stats['Period'] = monthly_stats['Year'].astype(str) + '-' + monthly_stats['Month'].astype(str).str.zfill(2)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Monthly total
ax1 = axes[0]
ax1.bar(monthly_stats['Period'], monthly_stats['sum'], color='steelblue', edgecolor='white')
ax1.set_title('Total Monthly DRM Keys')
ax1.set_ylabel('Total Keys')
ax1.tick_params(axis='x', rotation=45)

# Average daily by month (across years)
ax2 = axes[1]
month_avg = df.groupby('Month')['Total_Daily_Keys'].mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
colors_monthly = plt.cm.coolwarm(np.linspace(0.2, 0.8, 12))
ax2.bar(range(1, 13), month_avg.reindex(range(1, 13)).values, color=colors_monthly, edgecolor='white')
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(month_names)
ax2.set_title('Average Daily DRM Keys by Month')
ax2.set_ylabel('Avg Daily Keys')
ax2.axhline(df['Total_Daily_Keys'].mean(), color='crimson', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: weekday × month
heatmap_data = (
    df.groupby(['Month', 'DayOfWeek'])['Total_Daily_Keys']
    .mean()
    .unstack()
)
heatmap_data.columns = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
heatmap_data.index = month_names[:len(heatmap_data)]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg DRM Keys'}
)
ax.set_title('DRM Key Demand Heatmap — Weekday × Month')
ax.set_ylabel('Month')
ax.set_xlabel('Day of Week')
plt.tight_layout()
plt.show()

## 4. Source Breakdown — IPTV vs Fim+ vs BHD

In [ ]:
source_cols = [c for c in df_source.columns if c != 'LogDate']

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Stacked area chart
ax1 = axes[0]
ax1.stackplot(
    df_source['LogDate'],
    *[df_source[col].rolling(7, min_periods=1).mean() for col in source_cols],
    labels=source_cols, alpha=0.7
)
ax1.set_title('DRM Key Demand by Source (7-day MA, stacked)')
ax1.set_ylabel('DRM Keys')
ax1.legend(loc='upper left')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.tick_params(axis='x', rotation=45)

# Proportion over time
ax2 = axes[1]
df_pct = df_source[source_cols].div(df_source[source_cols].sum(axis=1), axis=0)
df_pct_smooth = df_pct.rolling(30, min_periods=1).mean()
for col in source_cols:
    ax2.plot(df_source['LogDate'], df_pct_smooth[col] * 100, label=col, linewidth=2)
ax2.set_title('Source Share Over Time (30-day MA, %)')
ax2.set_ylabel('Share (%)')
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Overall source split
total_by_source = df_source[source_cols].sum()
print('Overall source split:')
for src in source_cols:
    pct = total_by_source[src] / total_by_source.sum() * 100
    print(f'  {src:>10}: {total_by_source[src]:>12,.0f} ({pct:.1f}%)')

## 5. Time Series Decomposition

Decompose the series into trend, seasonal, and residual components.

In [ ]:
ts = df.set_index('LogDate')['Total_Daily_Keys']

# Weekly decomposition (period=7)
decomposition = seasonal_decompose(ts, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

decomposition.observed.plot(ax=axes[0], color='steelblue')
axes[0].set_title('Observed')
axes[0].set_ylabel('Keys')

decomposition.trend.plot(ax=axes[1], color='darkorange')
axes[1].set_title('Trend')
axes[1].set_ylabel('Keys')

decomposition.seasonal.plot(ax=axes[2], color='green')
axes[2].set_title('Weekly Seasonality (period=7)')
axes[2].set_ylabel('Keys')

decomposition.resid.plot(ax=axes[3], color='crimson', alpha=0.7)
axes[3].set_title('Residual')
axes[3].set_ylabel('Keys')

plt.suptitle('Additive Time Series Decomposition (Weekly)', y=1.01, fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal component — zoom into one representative week
seasonal_week = decomposition.seasonal[:7]

fig, ax = plt.subplots(figsize=(8, 4))
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors_bar = ['#4a90d9' if i >= 5 else '#95a5a6' for i in range(7)]
ax.bar(range(7), seasonal_week.values, color=colors_bar, edgecolor='white')
ax.set_xticks(range(7))
ax.set_xticklabels(day_labels)
ax.set_title('Weekly Seasonal Component (additive)')
ax.set_ylabel('Seasonal Effect (keys)')
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Stationarity Testing

Augmented Dickey-Fuller test to assess if the series is stationary (needed for many forecasting methods).

In [ ]:
def run_adf_test(series, name=''):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF Test — {name}')
    print(f'  Test Statistic : {result[0]:.4f}')
    print(f'  p-value        : {result[1]:.6f}')
    print(f'  Lags Used      : {result[2]}')
    print(f'  Observations   : {result[3]}')
    for key, val in result[4].items():
        print(f'  Critical ({key}): {val:.4f}')
    if result[1] < 0.05:
        print(f'  → STATIONARY (p < 0.05)\n')
    else:
        print(f'  → NON-STATIONARY (p >= 0.05)\n')

run_adf_test(ts, 'Original Series')
run_adf_test(ts.diff().dropna(), 'First Difference (d=1)')

## 7. Autocorrelation Analysis

ACF and PACF plots to identify lag structure and guide model parameter selection.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# ACF — original
plot_acf(ts.dropna(), lags=35, ax=axes[0, 0], title='ACF — Original Series')

# PACF — original
plot_pacf(ts.dropna(), lags=35, ax=axes[0, 1], title='PACF — Original Series', method='ywm')

# ACF — differenced
plot_acf(ts.diff().dropna(), lags=35, ax=axes[1, 0], title='ACF — First Difference')

# PACF — differenced
plot_pacf(ts.diff().dropna(), lags=35, ax=axes[1, 1], title='PACF — First Difference', method='ywm')

plt.tight_layout()
plt.show()

print('Key observations for model selection:')
print('  - Significant spike at lag 7 → strong weekly seasonality')
print('  - Use these patterns to set (p,d,q) and seasonal (P,D,Q,s) parameters')

## Summary of EDA Findings

| Finding | Detail | Impact on Forecasting |
|---|---|---|
| Weekly seasonality | Weekend demand significantly exceeds weekdays | Use `period=7` in seasonal models, add `IsWeekend` feature |
| Trend | [Observe from plots above] | May need differencing or trend component |
| Source composition | IPTV (K+/Đặc Sắc) vs Movie (Fim+/BHD) split | Consider separate forecasts if sources have different dynamics |
| Stationarity | [Check ADF results above] | Determines need for differencing |
| Autocorrelation | Strong lag-7 correlation | Supports SARIMA/Prophet weekly seasonality |

→ **Next:** `03_forecasting_model.ipynb` — train and evaluate forecasting models